<div style="text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);">
  <h1 style="color: white; margin: 0 0 8px 0; font-size: 2.5em;">🎙️ MOSS-TTS 1.7B - Zero-Shot Voice Cloning</h1>
  <h3 style="color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;">Google Colab T4 GPU Edition - Created by <strong>AIQUEST Academy</strong></h3>
  <p style="color: #ddd; margin: 0; text-align: center;">1.7B Parameter SOTA Discrete Audio Codec & Speech Generation | Powered by OpenMOSS</p>
</div>

<div align="center">
  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Colab-T4%20GPU-orange?style=for-the-badge&logo=googlecolab&logoColor=white" />
  <img src="https://img.shields.io/badge/Model-1.7B%20Params-green?style=for-the-badge" />
  <br><br>
  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://aiquest.site">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
  </a>
</div>

In [ ]:
#@title 📦 Step 1: Install Dependencies & Fast Setup
import sys
import os
import subprocess
import torch

print("=== Colab T4 Environment Setup ===")

# GPU Verification
if not torch.cuda.is_available():
    print("⚠️ WARNING: No GPU detected! Go to Runtime → Change runtime type → T4 GPU")
else:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU Detected: {gpu_name} ({vram_gb:.1f} GB VRAM)")

# Fast installation of only essential packages
print("\n📦 Installing optimized dependencies...")
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "transformers==5.17.0",  # pinned: Step 2 patches are verified against this exact release
        "accelerate>=1.2.0",
        "soundfile>=0.12.1",
        "gradio>=5.0.0",
        "librosa>=0.10.1"
    ],
    check=True
)

# Enable high-speed Xet-based downloader for ~13GB model weights
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

# Download official Cloudflare tunnel binary for secondary public access
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("🌐 Installing Cloudflare tunnel binary (cloudflared)...")
    try:
        subprocess.run(
            "curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
            shell=True,
            check=False
        )
        print("✅ Cloudflare tunnel (cloudflared) installed successfully!")
    except Exception as e:
        print(f"⚠️ Cloudflare tunnel notice: {e}")

print("✅ Dependencies installed & high-speed transfer enabled!")

In [ ]:
#@title 🚀 Step 2: Load MOSS-TTS 1.7B & Launch Gradio Interface
import os
# Eliminate memory fragmentation across autoregressive generation steps
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys
import gc
import re
import time
import random
import atexit
import warnings
import traceback
import subprocess
from datetime import datetime
from pathlib import Path

import torch
import torchaudio
import soundfile as sf
import librosa
import numpy as np
import gradio as gr
import inspect
import functools
from transformers import AutoModel, AutoProcessor
from transformers.generation import GenerationMixin
import transformers.masking_utils
from transformers.modeling_outputs import BaseModelOutputWithPast

# Suppress noisy warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Hardware and attention backend setup for Tesla T4
# Disable known broken cuDNN SDPA backend for MOSS-TTS as instructed by OpenMOSS
torch.backends.cuda.enable_cudnn_sdp(False)
torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

# Lossless speed-up: replay the local transformer as one CUDA graph (set False if you ever see a capture error)
USE_CUDA_GRAPH_LOCAL = True

# =========================================================================
# COMPATIBILITY PATCHES FOR MODERN TRANSFORMERS (>= 4.46 / 4.48+ / 5.0)
# =========================================================================

# Fix 1: Patch create_causal_mask to accept both 'input_embeds' (typo in MOSS-TTS) and 'inputs_embeds'
# and filter kwargs against the active transformers version's signature to prevent TypeError
_orig_create_causal_mask = transformers.masking_utils.create_causal_mask
_create_causal_mask_params = set(inspect.signature(_orig_create_causal_mask).parameters.keys())

def _patched_create_causal_mask(*args, **kwargs):
    if "input_embeds" in kwargs and "inputs_embeds" not in kwargs:
        kwargs["inputs_embeds"] = kwargs.pop("input_embeds")
    filtered_kwargs = {k: v for k, v in kwargs.items() if k in _create_causal_mask_params}
    return _orig_create_causal_mask(*args, **filtered_kwargs)

transformers.masking_utils.create_causal_mask = _patched_create_causal_mask

# Fix 2: Provide _get_initial_cache_position removed in newer transformers GenerationMixin
def _get_initial_cache_position(self, cur_len_or_ids, device_or_kwargs, model_kwargs=None):
    if model_kwargs is None:
        input_ids = cur_len_or_ids
        model_kwargs = device_or_kwargs
        cur_len = input_ids.shape[1] if hasattr(input_ids, "shape") else len(input_ids)
        dev = input_ids.device if hasattr(input_ids, "device") else "cpu"
    else:
        cur_len = cur_len_or_ids
        dev = device_or_kwargs

    if "cache_position" not in model_kwargs or model_kwargs["cache_position"] is None:
        past_length = 0
        if "past_key_values" in model_kwargs and model_kwargs["past_key_values"] is not None:
            pkv = model_kwargs["past_key_values"]
            if hasattr(pkv, "get_seq_length"):
                past_length = pkv.get_seq_length()
            elif hasattr(pkv, "get_usable_length"):
                past_length = pkv.get_usable_length(cur_len)
            elif isinstance(pkv, (tuple, list)) and len(pkv) > 0:
                past_length = pkv[0][0].shape[-2]
        model_kwargs["cache_position"] = torch.arange(past_length, cur_len, device=dev)
    return model_kwargs

GenerationMixin._get_initial_cache_position = _get_initial_cache_position

def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
    """
    Expand key/value states along head dimension for Grouped Query Attention (GQA).
    Transforms (batch, num_key_value_heads, seqlen, head_dim) -> (batch, num_attention_heads, seqlen, head_dim)
    """
    if n_rep == 1:
        return hidden_states
    return torch.repeat_interleave(hidden_states, dim=1, repeats=n_rep)

# Global model references
model = None
processor = None

def cleanup_model():
    """Unload model from GPU memory and flush cache"""
    global model, processor
    if model is not None:
        del model
        model = None
    if processor is not None:
        if hasattr(processor, "audio_tokenizer"):
            del processor.audio_tokenizer
        del processor
        processor = None
    if device == "cuda":
        torch.cuda.empty_cache()
        gc.collect()

atexit.register(cleanup_model)

def patch_moss_compatibility(loaded_model):
    """
    Apply runtime compatibility bridges:
    1. Bridge nested language_config attributes onto loaded_model.config for DynamicCache.
    2. Bind _get_initial_cache_position directly on model class.
    3. Ensure all loaded modules in sys.modules use patched create_causal_mask.
    """
    # Bridge config attributes for DynamicCache
    if hasattr(loaded_model.config, "language_config"):
        lang_cfg = loaded_model.config.language_config
        bridge_attrs = [
            "num_hidden_layers",
            "num_attention_heads",
            "num_key_value_heads",
            "sliding_window",
            "head_dim",
            "hidden_size",
            "vocab_size",
            "use_cache",
        ]
        for attr in bridge_attrs:
            if hasattr(lang_cfg, attr) and not hasattr(loaded_model.config, attr):
                setattr(loaded_model.config, attr, getattr(lang_cfg, attr))

    config_cls = loaded_model.config.__class__
    if not hasattr(config_cls, "num_hidden_layers"):
        config_cls.num_hidden_layers = property(
            lambda self: getattr(self.language_config, "num_hidden_layers", 28)
        )

    # Enable use_cache across configs
    loaded_model.config.use_cache = True
    if hasattr(loaded_model, "generation_config") and loaded_model.generation_config is not None:
        loaded_model.generation_config.use_cache = True

    # Bind cache position helper on model class
    loaded_model.__class__._get_initial_cache_position = _get_initial_cache_position

    # Patch create_causal_mask across all loaded modules
    for mod in list(sys.modules.values()):
        if hasattr(mod, "create_causal_mask"):
            try:
                mod.create_causal_mask = _patched_create_causal_mask
            except Exception:
                pass

    # Ensure SDPA always repeats KV heads on pre-Ampere GPUs (e.g. Tesla T4 Turing SM75)
    try:
        import transformers.integrations.sdpa_attention
        if hasattr(transformers.integrations.sdpa_attention, "use_gqa_in_sdpa"):
            transformers.integrations.sdpa_attention.use_gqa_in_sdpa = lambda *args, **kwargs: False
    except Exception:
        pass

    # Fix 3: Directly patch MossTTSLocalTransformer and MossTTSAttentionWithoutPositionalEmbedding
    # to enforce Grouped Query Attention (GQA) head alignment and bypass causal_mask calculation
    if hasattr(loaded_model, "local_transformer") and loaded_model.local_transformer is not None:
        local_tf = loaded_model.local_transformer
        local_tf_cls = local_tf.__class__

        # Patch self_attn on local_transformer layers to guarantee KV head repetition for GQA
        if hasattr(local_tf, "layers") and len(local_tf.layers) > 0 and hasattr(local_tf.layers[0], "self_attn"):
            attn_cls = local_tf.layers[0].self_attn.__class__

            def _safe_attn_forward(
                self,
                hidden_states: torch.Tensor,
                position_embeddings=None,
                attention_mask=None,
                past_key_value=None,
                cache_position=None,
                **kwargs,
            ):
                if hidden_states.dim() == 2:
                    hidden_states = hidden_states.unsqueeze(0)

                bsz, q_len, _ = hidden_states.shape
                num_heads = getattr(self, "num_heads", 16)
                num_kv_heads = getattr(self, "num_key_value_heads", 8)
                head_dim = getattr(self, "head_dim", 128)
                num_kv_groups = num_heads // num_kv_heads

                query_states = self.q_norm(self.q_proj(hidden_states).view(bsz, q_len, num_heads, head_dim)).transpose(1, 2)
                key_states = self.k_norm(self.k_proj(hidden_states).view(bsz, q_len, num_kv_heads, head_dim)).transpose(1, 2)
                value_states = self.v_proj(hidden_states).view(bsz, q_len, num_kv_heads, head_dim).transpose(1, 2)

                # Expand KV heads for Grouped Query Attention (GQA) compatibility on Tesla T4
                if num_kv_groups > 1:
                    key_states = key_states[:, :, None, :, :].expand(bsz, num_kv_heads, num_kv_groups, q_len, head_dim).reshape(bsz, num_heads, q_len, head_dim)
                    value_states = value_states[:, :, None, :, :].expand(bsz, num_kv_heads, num_kv_groups, q_len, head_dim).reshape(bsz, num_heads, q_len, head_dim)

                query_states = query_states.contiguous()
                key_states = key_states.contiguous()
                value_states = value_states.contiguous()

                try:
                    attn_output = torch.nn.functional.scaled_dot_product_attention(
                        query_states,
                        key_states,
                        value_states,
                        attn_mask=None,
                        dropout_p=0.0 if not self.training else self.attention_dropout,
                        is_causal=True,
                        scale=self.scaling,
                    )
                except Exception:
                    # Robust eager attention fallback for sequence length 33 on Tesla T4
                    attn_weights = torch.matmul(query_states, key_states.transpose(-2, -1)) * self.scaling
                    causal_mask = torch.triu(
                        torch.full((q_len, q_len), float("-inf"), device=query_states.device, dtype=query_states.dtype),
                        diagonal=1
                    )
                    attn_weights = attn_weights + causal_mask
                    attn_weights = torch.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)
                    attn_output = torch.matmul(attn_weights, value_states)

                attn_output = attn_output.transpose(1, 2).contiguous()
                attn_output = attn_output.reshape(bsz, q_len, -1)
                attn_output = self.o_proj(attn_output)
                return attn_output, None

            attn_cls.forward = _safe_attn_forward

        def _safe_local_forward(
            self,
            input_ids=None,
            attention_mask=None,
            position_ids=None,
            past_key_values=None,
            inputs_embeds=None,
            use_cache=None,
            output_attentions=None,
            output_hidden_states=None,
            return_dict=None,
            **kwargs
        ):
            output_attentions = output_attentions if output_attentions is not None else getattr(self.config, "output_attentions", False)
            output_hidden_states = (
                output_hidden_states if output_hidden_states is not None else getattr(self.config, "output_hidden_states", False)
            )
            use_cache = False
            return_dict = return_dict if return_dict is not None else getattr(self.config, "return_dict", True)

            if (input_ids is None) ^ (inputs_embeds is not None):
                raise ValueError("You cannot specify both input_ids and inputs_embeds at the same time, and must specify either one")

            if inputs_embeds is None:
                inputs_embeds = self.embed_tokens(input_ids)

            hidden_states = inputs_embeds
            if hidden_states.dim() == 2:
                hidden_states = hidden_states.unsqueeze(0)

            all_hidden_states = () if output_hidden_states else None
            all_self_attns = () if output_attentions else None
            next_decoder_cache = None

            for decoder_layer in self.layers:
                if output_hidden_states:
                    all_hidden_states += (hidden_states,)

                layer_outputs = decoder_layer(
                    hidden_states,
                    attention_mask=None,
                    position_ids=position_ids,
                    past_key_value=past_key_values,
                    output_attentions=output_attentions,
                    use_cache=False,
                )
                if isinstance(layer_outputs, (tuple, list)):
                    hidden_states = layer_outputs[0]
                else:
                    hidden_states = layer_outputs

            hidden_states = self.norm(hidden_states)

            if output_hidden_states:
                all_hidden_states += (hidden_states,)

            from transformers.modeling_outputs import BaseModelOutputWithPast
            if not return_dict:
                return tuple(v for v in [hidden_states, next_decoder_cache, all_hidden_states, all_self_attns] if v is not None)

            return BaseModelOutputWithPast(
                last_hidden_state=hidden_states,
                past_key_values=next_decoder_cache,
                hidden_states=all_hidden_states,
                attentions=all_self_attns,
            )

        local_tf_cls.forward = _safe_local_forward

    # Ensure _build_generation_config always sets use_cache=True for O(1) step latency
    if hasattr(loaded_model, "_build_generation_config"):
        orig_build_cfg = loaded_model._build_generation_config
        def _patched_build_generation_config(*args, **kwargs):
            kwargs.pop("use_cache", None)
            cfg = orig_build_cfg(*args, **kwargs)
            cfg.use_cache = True
            return cfg
        loaded_model._build_generation_config = _patched_build_generation_config

    # Fix 4 (ROOT CAUSE of gibberish audio): feed only the NEW frame to the global backbone once the KV cache is warm.
    # transformers 5.x (e.g. 5.17) only trims input_ids inside prepare_inputs_for_generation when the caller passes
    # `next_sequence_length`. MOSS-TTS's custom _sample() loop was written for transformers 5.0.0 (which trimmed via
    # cache_position) and never passes it. Result: every step re-fed the ENTIRE prompt + all generated frames on top of
    # the KV cache, with RoPE positions offset by the cache length. The backbone saw duplicated, mis-positioned context,
    # so the local transformer sampled valid-looking but meaningless RVQ codes (static / robotic babble).
    _orig_prepare = loaded_model.prepare_inputs_for_generation
    if "next_sequence_length" in inspect.signature(_orig_prepare).parameters:
        # functools.wraps keeps the original signature visible to generate()'s _validate_model_kwargs
        @functools.wraps(_orig_prepare)
        def _sliced_prepare_inputs_for_generation(input_ids, past_key_values=None, **kwargs):
            # Stale cache_position injected by _get_initial_cache_position is never advanced in 5.x - drop it
            kwargs.pop("cache_position", None)
            past_len = 0
            if past_key_values is not None and hasattr(past_key_values, "get_seq_length"):
                past_len = int(past_key_values.get_seq_length())
            if past_len > 0 and kwargs.get("next_sequence_length") is None:
                kwargs["next_sequence_length"] = input_ids.shape[1] - past_len
            return _orig_prepare(input_ids, past_key_values=past_key_values, **kwargs)
        loaded_model.prepare_inputs_for_generation = _sliced_prepare_inputs_for_generation

    # Fix 5: fp16 overflow guard for MossTTSRMSNorm (layer_norm_before_lm_heads).
    # Upstream runs in bfloat16 and computes x.pow(2) without upcasting. In float16 any activation above ~256 squares to
    # inf, the norm collapses to 0, the logits go flat and sampling becomes uniform noise. Compute the norm in float32.
    if hasattr(loaded_model, "layer_norm_before_lm_heads") and len(loaded_model.layer_norm_before_lm_heads) > 0:
        rms_cls = loaded_model.layer_norm_before_lm_heads[0].__class__

        def _fp32_rmsnorm_forward(self, x):
            in_dtype = x.dtype
            xf = x.float()
            xf = xf * torch.rsqrt(xf.pow(2).mean(dim=-1, keepdim=True) + self.eps)
            return (xf * self.weight.float()).to(in_dtype)

        rms_cls.forward = _fp32_rmsnorm_forward

    # Speed-up (lossless): CUDA-graph the local transformer.
    # Every audio frame runs the 4-layer local transformer up to 33 times in a row, and each run launches ~100 tiny
    # kernels, so a T4 mostly waits on Python / kernel-launch overhead instead of doing math. We run it on a fixed
    # 33-slot buffer and replay one captured CUDA graph instead. Attention is causal, so the unused tail slots cannot
    # influence the real positions: outputs match eager mode (up to fp rounding). Same RVQ depth, same sampling.
    local_tf = getattr(loaded_model, "local_transformer", None)
    if USE_CUDA_GRAPH_LOCAL and device == "cuda" and local_tf is not None:
        eager_local_forward = functools.partial(type(local_tf).forward, local_tf)
        max_local_len = int(getattr(loaded_model, "channels", 33))
        graph_state = {"graph": None, "static_in": None, "static_out": None, "failed": False}

        def _capture_local_graph(x):
            static_in = torch.zeros(1, max_local_len, x.shape[-1], dtype=x.dtype, device=x.device)
            # Warm up on a side stream (required before capture)
            side = torch.cuda.Stream()
            side.wait_stream(torch.cuda.current_stream())
            with torch.cuda.stream(side):
                for _ in range(3):
                    eager_local_forward(inputs_embeds=static_in)
            torch.cuda.current_stream().wait_stream(side)
            graph = torch.cuda.CUDAGraph()
            with torch.cuda.graph(graph):
                static_out = eager_local_forward(inputs_embeds=static_in).last_hidden_state
            graph_state.update(graph=graph, static_in=static_in, static_out=static_out)
            print("⚡ Local transformer CUDA graph captured (lossless speed-up active)")

        def _graphed_local_forward(input_ids=None, attention_mask=None, inputs_embeds=None, **kwargs):
            x = inputs_embeds
            usable = (
                x is not None and x.is_cuda and x.dim() == 3
                and x.shape[0] == 1 and x.shape[1] <= max_local_len
                and not graph_state["failed"]
            )
            if usable and graph_state["graph"] is None:
                try:
                    _capture_local_graph(x)
                except Exception as e:
                    graph_state["failed"] = True
                    usable = False
                    print(f"⚠️ CUDA graph capture failed ({type(e).__name__}: {e}) - falling back to eager local transformer")
            if usable and x.dtype == graph_state["static_in"].dtype:
                t = x.shape[1]
                graph_state["static_in"][:, :t].copy_(x)
                graph_state["graph"].replay()
                # clone: static_out is overwritten on the next replay
                return BaseModelOutputWithPast(last_hidden_state=graph_state["static_out"][:, :t].clone())
            return eager_local_forward(input_ids=input_ids, attention_mask=attention_mask, inputs_embeds=inputs_embeds, **kwargs)

        local_tf.forward = _graphed_local_forward

    # Attach step sentinel onto model.forward
    step_counter = 0
    orig_forward = loaded_model.forward
    @functools.wraps(orig_forward)
    def _mem_safe_forward(*args, **kwargs):
        nonlocal step_counter
        step_counter += 1
        # One-time sanity check: after the prefill step, each decode step must feed exactly 1 new frame
        if step_counter == 2:
            ids = kwargs.get("input_ids", args[0] if args else None)
            if ids is not None and hasattr(ids, "shape"):
                n_new = int(ids.shape[1])
                print(f"{'✅' if n_new == 1 else '❌'} KV-cache decode step feeds {n_new} frame(s) (expected 1)")
        # No per-step torch.cuda.empty_cache() here: it forces a GPU sync + re-allocation every few frames (slow) and
        # is not needed now that the KV cache grows by exactly 1 frame per step (~0.11 MB/frame, ~1.1 GB at 10k).
        return orig_forward(*args, **kwargs)
    loaded_model.forward = _mem_safe_forward

def load_model():
    """Load processor and model with compatibility patches"""
    global model, processor

    if model is None:
        print("🔄 Loading MOSS-TTS 1.7B (high-speed transfer active)...")
        start_time = time.time()

        processor = AutoProcessor.from_pretrained(
            "OpenMOSS-Team/MOSS-TTS-Local-Transformer",
            trust_remote_code=True,
        )

        # Offload audio_tokenizer strictly and permanently to System RAM (CPU)
        if hasattr(processor, "audio_tokenizer") and processor.audio_tokenizer is not None:
            processor.audio_tokenizer = processor.audio_tokenizer.to("cpu")

        if device == "cuda":
            torch.cuda.empty_cache()
            gc.collect()

        model = AutoModel.from_pretrained(
            "OpenMOSS-Team/MOSS-TTS-Local-Transformer",
            trust_remote_code=True,
            attn_implementation="sdpa" if device == "cuda" else "eager",
            dtype=dtype,
            low_cpu_mem_usage=True,
        ).to(device)

        # Apply complete compatibility patches
        patch_moss_compatibility(model)

        model.eval()

        if device == "cuda":
            torch.cuda.empty_cache()
            gc.collect()
            vram = torch.cuda.memory_allocated() / (1024**3)
            load_elapsed = time.time() - start_time
            print(f"✅ Model loaded in {load_elapsed:.1f}s! VRAM: {vram:.2f}GB (Codec on System RAM)")

    return model, processor

LANGUAGES = [
    "English", "Chinese", "Japanese", "Korean", "French", "German",
    "Spanish", "Italian", "Portuguese", "Russian", "Arabic", "Cantonese",
    "Vietnamese", "Thai", "Turkish", "Hindi", "Indonesian", "Malay",
    "Dutch", "Swedish", "Polish", "Danish", "Finnish", "Norwegian",
    "Czech", "Greek", "Hungarian", "Romanian", "Slovak", "Ukrainian",
    "Hebrew", "Auto"
]

PRESETS = {
    "Studio Master (32 RVQ) - Recommended": {
        "n_vq": 32,
        "text_temp": 1.0,
        "audio_temp": 1.0,
        "text_top_p": 0.95,
        "audio_top_p": 0.95,
        "text_top_k": 50,
        "audio_top_k": 50,
        "audio_rep_pen": 1.1
    },
    "High Quality (24 RVQ) - Fast": {
        "n_vq": 24,
        "text_temp": 1.0,
        "audio_temp": 1.0,
        "text_top_p": 0.95,
        "audio_top_p": 0.95,
        "text_top_k": 50,
        "audio_top_k": 50,
        "audio_rep_pen": 1.1
    },
    "Balanced (16 RVQ) - Lightweight": {
        "n_vq": 16,
        "text_temp": 0.9,
        "audio_temp": 0.9,
        "text_top_p": 0.95,
        "audio_top_p": 0.95,
        "text_top_k": 50,
        "audio_top_k": 50,
        "audio_rep_pen": 1.1
    },
    "Fast (8 RVQ) - Draft Mode": {
        "n_vq": 8,
        "text_temp": 0.8,
        "audio_temp": 0.85,
        "text_top_p": 0.95,
        "audio_top_p": 0.95,
        "text_top_k": 50,
        "audio_top_k": 50,
        "audio_rep_pen": 1.1
    }
}

def apply_preset(preset_name):
    """Return preset configuration values"""
    preset = PRESETS[preset_name]
    return (
        preset["n_vq"],
        preset["text_temp"],
        preset["text_top_p"],
        preset["text_top_k"],
        preset["audio_temp"],
        preset["audio_top_p"],
        preset["audio_top_k"],
        preset["audio_rep_pen"]
    )

def generate_speech(
    text,
    reference_audio,
    language,
    max_new_tokens,
    speed,
    text_temp,
    text_top_p,
    text_top_k,
    audio_temp,
    audio_top_p,
    audio_top_k,
    audio_repetition_penalty,
    n_vq,
    seed=-1,
    progress=gr.Progress()
):
    """Generate high-fidelity speech with zero-shot voice cloning and real-time console logs"""
    if not text or len(text.strip()) == 0:
        return None, "⚠️ Please enter text to generate speech."

    try:
        os.makedirs("outputs", exist_ok=True)

        progress(0.05, desc="Initializing model...")
        curr_model, curr_processor = load_model()

        text_length = len(text)
        estimated_duration = max_new_tokens / 12.5
        mode_str = f"Voice Cloning ({os.path.basename(reference_audio)})" if (reference_audio and len(str(reference_audio).strip()) > 0) else "Default Base Voice"

        # Console logging for notebook output
        print(f"\n{'='*55}")
        print(f"🎤 [MOSS-TTS 1.7B] New Speech Generation Request")
        print(f"📝 Script Length: {text_length:,} characters")
        print(f"🎯 Target Tokens: {max_new_tokens} (~{estimated_duration/60:.1f} min est.)")
        print(f"🎙️ Mode: {mode_str}")
        print(f"🗣️ Language: {language}")
        print(f"🎛️ RVQ Depth: {int(n_vq)}/32 layers | Playback Speed: {speed:.1f}x")
        print(f"{'='*55}")

        status = f"📝 Script: {text_length:,} characters\n"
        status += f"🎯 Target Tokens: {max_new_tokens} (~{estimated_duration/60:.1f} min)\n"
        status += f"🗣️ Language: {language}\n"

        # Build conversation representation with automatic duration inspection and trimming
        progress(0.15, desc="Building input representation...")
        ref_audio_path = None
        if reference_audio is not None and len(str(reference_audio).strip()) > 0:
            if os.path.exists(reference_audio):
                try:
                    dur_s = None
                    try:
                        sf_info = sf.info(reference_audio)
                        dur_s = sf_info.duration
                    except Exception:
                        dur_s = librosa.get_duration(path=reference_audio)

                    if dur_s is not None and dur_s > 10.0:
                        print(f"✂️ Reference audio is {dur_s:.1f}s. Automatically trimming to first 10.0s for optimal voice cloning and VRAM safety...")
                        status += f"✂️ Reference audio trimmed to first 10.0s (was {dur_s:.1f}s)\n"
                        y, sr_loaded = librosa.load(reference_audio, sr=None, duration=10.0)
                        trimmed_ref_path = "outputs/trimmed_ref_audio.wav"
                        sf.write(trimmed_ref_path, y, sr_loaded)
                        ref_audio_path = trimmed_ref_path
                    else:
                        ref_audio_path = reference_audio
                except Exception as e:
                    print(f"⚠️ Audio inspect notice: {e}")
                    ref_audio_path = reference_audio
            else:
                ref_audio_path = None

        if ref_audio_path:
            status += f"🎙️ Mode: Voice Cloning ({os.path.basename(ref_audio_path)})\n"
            user_kwargs = {"text": text, "reference": [ref_audio_path]}
            if language and language != "Auto":
                user_kwargs["language"] = language
            conversations = [[
                curr_processor.build_user_message(**user_kwargs)
            ]]
        else:
            status += "🎙️ Mode: Default Base Voice\n"
            user_kwargs = {"text": text}
            if language and language != "Auto":
                user_kwargs["language"] = language
            conversations = [[
                curr_processor.build_user_message(**user_kwargs)
            ]]

        yield None, status

        # Tokenize inputs on CPU (Audio Tokenizer runs strictly on System RAM)
        progress(0.25, desc="Tokenizing inputs on CPU...")
        if hasattr(curr_processor, "audio_tokenizer") and curr_processor.audio_tokenizer is not None:
            curr_processor.audio_tokenizer.to("cpu")

        batch = curr_processor(conversations, mode="generation")
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # Avoid exact 1.0 temperature edge cases
        if text_temp == 1.0:
            text_temp = 1.001
        if audio_temp == 1.0:
            audio_temp = 1.001

        if device == "cuda":
            torch.cuda.empty_cache()
            gc.collect()

        status += f"🎛️ RVQ Depth: {int(n_vq)}/32 layers\n"
        status += "⚡ Generating speech latents...\n"
        print(f"⚡ Synthesizing speech latents (RVQ {int(n_vq)})...")
        yield None, status

        # Seed: -1 = new random seed every run; any other value reproduces the same take
        try:
            seed = int(seed)
        except (TypeError, ValueError):
            seed = -1
        if seed < 0:
            seed = int.from_bytes(os.urandom(4), "little") & 0x7FFFFFFF
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)  # also seeds all CUDA devices
        print(f"🎲 Seed: {seed}")
        status += f"🎲 Seed: {seed}\n"

        # Execute generation with KV cache enabled for O(1) step latency & memory
        start_gen = time.time()
        progress(0.35, desc="Synthesizing audio...")

        with torch.inference_mode():
            outputs = curr_model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=int(max_new_tokens),
                text_temperature=float(text_temp),
                text_top_p=float(text_top_p),
                text_top_k=int(text_top_k),
                audio_temperature=float(audio_temp),
                audio_top_p=float(audio_top_p),
                audio_top_k=int(audio_top_k),
                audio_repetition_penalty=float(audio_repetition_penalty),
                n_vq_for_inference=int(n_vq),
            )

        gen_time = time.time() - start_gen

        # Clean up generation inputs immediately to free VRAM for decoding
        if device == "cuda":
            del input_ids, attention_mask, batch
            torch.cuda.empty_cache()
            gc.collect()

        progress(0.85, desc="Decoding waveform on CPU...")
        status += f"✅ Latents synthesized in {gen_time:.2f}s\n"
        status += "🔊 Neural audio codec decoding (System RAM)...\n"
        print(f"✅ Latents synthesized in {gen_time:.2f}s! Decoding waveform on CPU...")
        yield None, status

        # Decode discrete audio codes strictly on CPU (System RAM) to eliminate GPU VRAM pressure
        if hasattr(curr_processor, "audio_tokenizer") and curr_processor.audio_tokenizer is not None:
            curr_processor.audio_tokenizer.to("cpu")

        # Safely move outputs to CPU before decoding to eliminate GPU memory pressure
        def _to_cpu(data):
            if isinstance(data, torch.Tensor):
                return data.to("cpu")
            elif isinstance(data, (list, tuple)):
                return type(data)(_to_cpu(item) for item in data)
            elif isinstance(data, dict):
                return {k: _to_cpu(v) for k, v in data.items()}
            return data

        outputs_cpu = _to_cpu(outputs)

        # Keep only the RVQ codebooks that were actually sampled. For n_vq < 32 the generator fills the unused
        # codebooks with code 0 (not pad), and the codec would otherwise decode those as real residuals (hiss).
        n_vq_used = int(n_vq)
        outputs_cpu = [(start_len, gen_ids[:, : 1 + n_vq_used]) for start_len, gen_ids in outputs_cpu]

        decoded_messages = curr_processor.decode(outputs_cpu)

        if not decoded_messages or not getattr(decoded_messages[0], "audio_codes_list", None) or len(decoded_messages[0].audio_codes_list) == 0:
            err_msg = "❌ Error: Model completed generation but returned empty audio codes. Please check text or try different sampling settings."
            yield None, err_msg
            return

        audio = decoded_messages[0].audio_codes_list[0]

        # Explicit cleanup of outputs
        if device == "cuda":
            del outputs, outputs_cpu, decoded_messages
            torch.cuda.empty_cache()
            gc.collect()

        sample_rate = int(curr_processor.model_config.sampling_rate)

        # Apply speed adjustment if requested
        if abs(speed - 1.0) > 1e-3:
            progress(0.92, desc="Adjusting audio speed...")
            audio = torchaudio.functional.resample(
                audio.unsqueeze(0),
                orig_freq=int(sample_rate * speed),
                new_freq=sample_rate
            ).squeeze(0)

        # Peak normalization to prevent digital clipping and ensure clean volume
        if isinstance(audio, torch.Tensor):
            max_amp = torch.max(torch.abs(audio)).item()
            if max_amp > 1e-6:
                audio = (audio / max_amp) * 0.95
            audio_np = audio.cpu().numpy()
        else:
            max_amp = float(np.max(np.abs(audio)))
            if max_amp > 1e-6:
                audio = (audio / max_amp) * 0.95
            audio_np = audio

        progress(0.96, desc="Saving audio file...")
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = f"outputs/moss_tts_{timestamp}.wav"
        sf.write(output_path, audio_np, sample_rate)

        duration = len(audio_np) / sample_rate
        rtf = gen_time / duration if duration > 0 else 0
        vram = torch.cuda.memory_allocated() / (1024**3) if device == "cuda" else 0

        # Stream complete metrics to notebook console
        print(f"🎉 SUCCESS! Audio Duration: {duration:.1f}s ({duration/60:.2f} min)")
        print(f"⏱️ Generation Time: {gen_time:.2f}s | Real-Time Factor (RTF): {rtf:.2f}x")
        print(f"📊 Active VRAM: {vram:.2f} GB | Saved: {output_path}")
        print(f"{'='*55}\n")

        progress(1.0, desc="Completed!")
        status += "\n🎉 Generation Successful!\n"
        status += f"📏 Audio Duration: {duration:.1f}s ({duration/60:.2f} min)\n"
        status += f"⏱️ Generation Time: {gen_time:.1f}s ({gen_time/60:.1f} min)\n"
        status += f"🚀 Real-Time Factor (RTF): {rtf:.2f}x\n"
        status += f"🎚️ Playback Speed: {speed:.1f}x\n"
        status += f"📊 Active VRAM: {vram:.2f} GB\n"
        status += f"💾 Saved to: {output_path}"

        yield output_path, status

    except torch.cuda.OutOfMemoryError:
        err_msg = "❌ GPU OUT OF MEMORY!\n\n"
        err_msg += f"Attempted {max_new_tokens} tokens with {n_vq} RVQ layers.\n\n"
        err_msg += "Recommended Actions:\n"
        err_msg += "1. Switch to 'Fast (8 RVQ)' or 'Balanced (16 RVQ)' preset\n"
        err_msg += "2. Reduce Max Tokens slider\n"
        err_msg += "3. Click 'Clear' to flush memory and retry"
        print(f"\n[MOSS-TTS OOM ERROR] {err_msg}\n")
        if device == "cuda":
            torch.cuda.empty_cache()
            gc.collect()
        yield None, err_msg
    except Exception as e:
        err_msg = f"❌ Error: {str(e)}\n\n{traceback.format_exc()}"
        print(f"\n[MOSS-TTS ERROR] {err_msg}\n")
        yield None, err_msg

# =============================================
# AIQUEST ACADEMY BRANDED GRADIO INTERFACE
# =============================================

custom_css = """
* { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.social-buttons { display: flex; justify-content: center; gap: 12px; flex-wrap: wrap; }
.social-btn { padding: 10px 24px; border-radius: 8px; font-weight: 700; font-size: 15px; text-decoration: none; display: inline-block; color: white !important; transition: all 0.3s; box-shadow: 0 4px 12px rgba(0,0,0,0.2); }
.social-btn:hover { transform: translateY(-2px); box-shadow: 0 6px 16px rgba(0,0,0,0.3); }
.youtube-btn { background: linear-gradient(135deg, #FF0000 0%, #CC0000 100%); }
.x-btn { background: linear-gradient(135deg, #000000 0%, #333333 100%); }
#gen-btn { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#stop-btn { background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#clear-btn { background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
.footer { text-align: center; padding: 20px; margin-top: 30px; border-top: 2px solid #e5e7eb; color: #6b7280; font-size: 0.9em; }
"""

with gr.Blocks(
    title="MOSS-TTS 1.7B by AIQUEST Academy",
    theme=gr.themes.Default(),
    css=custom_css
) as demo:

    gr.HTML("""
    <div class="brand-header">
        <div class="brand-title">🎙️ MOSS-TTS 1.7B - Zero-Shot Voice Cloning</div>
        <div class="brand-subtitle">1.7B Parameter SOTA Speech Synthesis - Optimized for Colab Free Tier (T4 GPU)</div>
        <div class="social-buttons">
            <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn youtube-btn">
                ▶ Subscribe on YouTube
            </a>
            <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">
                Follow on X
            </a>
        </div>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            text_input = gr.Textbox(
                label="📝 Text Input",
                placeholder="Type or paste text script here...",
                lines=8,
                value="Hello! This is MOSS text-to-speech, running on Google Colab free tier. Notebook by AIQUEST Academy."
            )

            language_dropdown = gr.Dropdown(
                choices=LANGUAGES,
                value="English",
                label="🗣️ Speech Language"
            )

            reference_audio = gr.Audio(
                label="🎤 Reference Audio (Optional - upload 3-10s clip for voice cloning)",
                type="filepath",
                sources=["upload"]
            )

            preset_dropdown = gr.Dropdown(
                choices=list(PRESETS.keys()),
                value="Studio Master (32 RVQ) - Recommended",
                label="Quality & Speed Preset"
            )

            with gr.Row():
                max_tokens = gr.Slider(
                    50, 10000, 10000, step=50,
                    label="Max Tokens (12.5 = 1s of audio)"
                )
                speed = gr.Slider(
                    0.5, 2.0, 1.0, step=0.1,
                    label="Playback Speed"
                )
                seed_input = gr.Number(
                    value=-1, precision=0,
                    label="🎲 Seed (-1 = random)"
                )

            with gr.Accordion("⚙️ Advanced Hyperparameters", open=False):
                n_vq = gr.Slider(8, 32, 32, step=1, label="RVQ Depth Layers")
                with gr.Row():
                    text_temp = gr.Slider(0.1, 1.5, 1.0, step=0.05, label="Text Temperature")
                    text_top_p = gr.Slider(0.1, 1.0, 0.95, step=0.05, label="Text Top-P")
                    text_top_k = gr.Slider(1, 100, 50, step=1, label="Text Top-K")
                with gr.Row():
                    audio_temp = gr.Slider(0.1, 1.5, 1.0, step=0.05, label="Audio Temperature")
                    audio_top_p = gr.Slider(0.1, 1.0, 0.95, step=0.05, label="Audio Top-P")
                    audio_top_k = gr.Slider(1, 100, 50, step=1, label="Audio Top-K")
                with gr.Row():
                    audio_rep_pen = gr.Slider(1.0, 1.5, 1.1, step=0.05, label="Audio Repetition Penalty")

            # Standard AIQUEST 3-Button Row
            with gr.Row():
                gen_btn = gr.Button("🎬 Generate Speech", variant="primary", size="lg", scale=3, elem_id="gen-btn")
                stop_btn = gr.Button("🛑 Stop", variant="secondary", size="lg", scale=1, elem_id="stop-btn")
                clear_btn = gr.Button("🗑️ Clear", variant="secondary", size="lg", scale=1, elem_id="clear-btn")

        with gr.Column(scale=1):
            audio_output = gr.Audio(label="🔊 Generated Speech Audio", type="filepath")
            status_output = gr.Textbox(label="📊 Generation Status", lines=16, interactive=False)

    gr.HTML("""
    <div class="footer">
        ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> &nbsp;|&nbsp;
        <a href="https://aiquest.site" target="_blank" style="color: #667eea; text-decoration: none; font-weight: 600;">aiquest.site</a> &nbsp;|&nbsp;
        <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" style="color: #FF0000; text-decoration: none; font-weight: 600;">YouTube</a> &nbsp;|&nbsp;
        <a href="https://x.com/aiquestacademy" target="_blank" style="color: #111827; text-decoration: none; font-weight: 600;">X (Twitter)</a>
        <br><span style="opacity: 0.8; font-size: 0.85em; margin-top: 4px; display: inline-block;">All rights reserved © AIQUEST Academy</span>
    </div>
    """)

    # Interactive Event Handlers
    preset_dropdown.change(
        fn=apply_preset,
        inputs=[preset_dropdown],
        outputs=[n_vq, text_temp, text_top_p, text_top_k,
                 audio_temp, audio_top_p, audio_top_k, audio_rep_pen]
    )

    gen_event = gen_btn.click(
        fn=generate_speech,
        inputs=[
            text_input, reference_audio, language_dropdown, max_tokens, speed,
            text_temp, text_top_p, text_top_k,
            audio_temp, audio_top_p, audio_top_k,
            audio_rep_pen, n_vq, seed_input
        ],
        outputs=[audio_output, status_output]
    )

    stop_btn.click(fn=None, cancels=[gen_event])

    def on_clear():
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()
        return (
            "Hello! This is MOSS text-to-speech, running on Google Colab free tier. Notebook by AIQUEST Academy.",
            "English",
            None,
            "Studio Master (32 RVQ) - Recommended",
            10000,
            1.0,
            -1,
            None,
            "🗑️ Reset inputs and cleared GPU cache."
        )

    clear_btn.click(
        fn=on_clear,
        inputs=[],
        outputs=[text_input, language_dropdown, reference_audio, preset_dropdown, max_tokens, speed, seed_input, audio_output, status_output]
    )

def start_cloudflare_tunnel(port=7860):
    """
    Start Cloudflare tunnel in background and extract public trycloudflare.com URL.
    Returns: (subprocess.Popen or None, tunnel_url or None)
    """
    cf_bin = "/usr/local/bin/cloudflared"
    if not os.path.exists(cf_bin):
        import shutil
        cf_bin = shutil.which("cloudflared")

    if not cf_bin or not os.path.exists(cf_bin):
        return None, None

    log_file = "/tmp/cloudflared.log"
    try:
        log_f = open(log_file, "w")
        proc = subprocess.Popen(
            [cf_bin, "tunnel", "--url", f"http://127.0.0.1:{port}"],
            stdout=log_f,
            stderr=subprocess.STDOUT
        )
    except Exception as e:
        print(f"⚠️ Could not start Cloudflare tunnel: {e}")
        return None, None

    # Poll log file for trycloudflare.com link
    tunnel_url = None
    for _ in range(30):
        time.sleep(0.5)
        if os.path.exists(log_file):
            try:
                with open(log_file, "r") as f:
                    content = f.read()
                    matches = re.findall(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", content)
                    if matches:
                        tunnel_url = matches[0]
                        break
            except Exception:
                pass

    return proc, tunnel_url

# Pre-load model at startup before launch
print("🔄 Initializing MOSS-TTS 1.7B model weights...")
load_model()

SERVER_PORT = 7860
print("\n" + "=" * 60)
print("🚀 Launching MOSS-TTS 1.7B Gradio Web Server & Cloudflare Tunnel...")
print("=" * 60)

# Launch Gradio without embedded notebook iframe
launch_res = demo.launch(
    share=True,
    inline=False,
    server_port=SERVER_PORT,
    prevent_thread_lock=True
)

share_url = getattr(demo, "share_url", None)
if not share_url and isinstance(launch_res, tuple) and len(launch_res) >= 3:
    share_url = launch_res[2]

# Launch Cloudflare Tunnel in background
cf_proc, cf_url = start_cloudflare_tunnel(port=SERVER_PORT)

print("\n" + "=" * 60)
print("🎙️ MOSS-TTS 1.7B Web Interface is Live!")
print("=" * 60)
if share_url:
    print(f"🔗 Gradio Public Share:     {share_url}")
if cf_url:
    print(f"🌐 Cloudflare Tunnel URL:   {cf_url}")
elif cf_proc is None:
    print("🌐 Cloudflare Tunnel:       (Binary not installed in Step 1)")
else:
    print("🌐 Cloudflare Tunnel:       (Connecting... check /tmp/cloudflared.log)")
print(f"🖥️ Local Instance URL:      http://127.0.0.1:{SERVER_PORT}")
print("=" * 60)
print("💡 Tip: Use the Cloudflare URL if Gradio share link ever lags or disconnects.")
print("=" * 60 + "\n")

# Keep the server running and stream incoming generation logs in real time
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Shutting down server...")
    if cf_proc is not None:
        cf_proc.terminate()

---
<div align="center">
  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://aiquest.site">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
  </a>
</div>
<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px; text-align:center;">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved
</p>
---